# Agriculture Data Cleaning: USDA-NASS Crop Yields

Cleans the USDA-NASS Quick Stats **county-level crop** export for Iowa into a
tidy, analysis-ready table: one row per
`(program, year, county, commodity_detail, statistic)`.

**Input:**  `data/tabular/01_raw/agriculture/Crop-Yields.csv`
**Output:** `data/tabular/02_clean/agriculture/crop-yields-clean.csv`

**What's in it** — 7 commodities (corn, soybeans, hay, oats, wheat, barley, rye)
reported as `PRODUCTION`, `YIELD`, or `ACRES PLANTED`, for 2015-2025, from both
the `SURVEY` and `CENSUS` programs. The wide-and-opaque `Data Item` string
(e.g. `CORN, GRAIN - YIELD, MEASURED IN BU / ACRE`) carries the commodity,
statistic, and unit packed together; we unpack it into explicit columns.

**Pipeline**
1. **Load** the raw 21-column export as strings (codes, not arithmetic, until parsed).
2. **Drop dead weight** — columns that are entirely empty or constant for this
   Iowa crop slice carry no information.
3. **Parse the join keys & values** — `year` to int, a 5-digit `county_fips`,
   and `value`/`cv_pct` from NASS text (suppression codes -> `NaN`, flagged).
4. **Unpack `Data Item`** into `commodity_detail` / `statistic` / `unit`.
5. **Enforce the key** is unique (the `OTHER (COMBINED) COUNTIES` rollups repeat
   across ag districts, so `ag_district_code` is part of the key).
6. **Sanity check** and **save**.

In [ ]:
import re
import numpy as np
import pandas as pd
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    """Walk upward until we find the repo's data/tabular directory.

    Notebooks have no __file__, and the kernel's working directory varies, so
    resolving paths relative to a fixed number of "../" is fragile. Searching
    upward for a sentinel makes the notebook runnable from anywhere.
    """
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "data" / "tabular").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate repo root containing data/tabular/")


REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / "data" / "tabular" / "01_raw" / "agriculture"
CLEAN_DIR = REPO_ROOT / "data" / "tabular" / "02_clean" / "agriculture"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO_ROOT)
print("Raw dir:  ", RAW_DIR)
print("Clean dir:", CLEAN_DIR)

In [ ]:
# --- USDA-NASS shared cleaning helpers ------------------------------------
#
# NASS uses parenthetical letter codes in place of numbers. They are NOT data;
# they encode *why* a number is absent, so they must become NaN (never 0):
#   (D) withheld to avoid disclosing data for individual operations
#   (Z) value rounds to less than half the unit shown
#   (X) not applicable
#   (NA) not available
#   (H) sampling CV >= 99.95% (estimate too unreliable to publish)
#   (L) sampling CV  <  0.05%
# (D) appears in `Value`; (D)/(H)/(L) appear in `CV (%)`.
NASS_SUPPRESSION = {"(D)", "(Z)", "(X)", "(NA)", "(H)", "(L)", "(NA)", "(S)"}


def parse_nass_numeric(series: pd.Series) -> tuple[pd.Series, pd.Series]:
    """Parse a NASS Value/CV column into (float, was_suppressed_flag).

    Strips thousands separators, maps every suppression code to NaN, and flags
    which rows were a real suppression code (vs. genuinely blank) so downstream
    users can tell "censored" apart from "not collected".
    """
    s = series.astype("string").str.strip()
    suppressed = s.isin(NASS_SUPPRESSION)
    cleaned = s.mask(s.isin(NASS_SUPPRESSION))           # codes -> <NA>
    cleaned = cleaned.str.replace(",", "", regex=False)  # 1,234 -> 1234
    return pd.to_numeric(cleaned, errors="coerce"), suppressed.fillna(False)


# A Data Item is "<COMMODITY DETAIL> - <STATISTIC>[, MEASURED IN <UNIT>]",
# e.g. "CORN, GRAIN - YIELD, MEASURED IN BU / ACRE" or "CORN - ACRES PLANTED".
_DATA_ITEM_RE = re.compile(r"^(?P<detail>.+?) - (?P<stat>.+?)(?:, MEASURED IN (?P<unit>.+))?$")


def parse_data_item(item: str) -> tuple[str, str, str | None]:
    """Split a Data Item string into (commodity_detail, statistic, unit)."""
    m = _DATA_ITEM_RE.match(item)
    if not m:
        raise ValueError(f"Unparseable Data Item: {item!r}")
    return m.group("detail"), m.group("stat"), m.group("unit")

## Step 1 - Load

Read everything as strings; Value/CV/ANSI are categorical text we parse deliberately below.

In [ ]:
RAW_FILE = "Crop-Yields.csv"
raw = pd.read_csv(RAW_DIR / RAW_FILE, dtype="string")
n_raw = len(raw)
print(f"Loaded {n_raw:,} rows x {raw.shape[1]} cols from {RAW_FILE}")
raw.head()

In [ ]:
# Every Data Item must parse, and every State ANSI must be Iowa (19) -- guard
# against a future re-pull silently changing the schema or scope.
assert raw["State ANSI"].dropna().eq("19").all(), "Non-Iowa rows present!"
_unparsed = [it for it in raw["Data Item"].dropna().unique() if not _DATA_ITEM_RE.match(it)]
assert not _unparsed, f"Unparseable Data Items: {_unparsed}"
print("Schema guards passed: all Iowa, all Data Items parse.")

## Step 2 - Drop structurally-empty and constant columns

For this Iowa county-crop slice several columns are entirely blank
(`Week Ending`, `Zip Code`, `Region`, `Watershed`) or hold a single constant
value (`Period=YEAR`, `Geo Level=COUNTY`, `State=IOWA`, `Domain=TOTAL`,
`Domain Category=NOT SPECIFIED`, `watershed_code`). They carry no signal, so we
drop them after verifying they really are constant/empty — keeping `State ANSI`
only long enough to build the FIPS key.

In [ ]:
EMPTY_COLS = ["Week Ending", "Zip Code", "Region", "Watershed"]
CONST_COLS = ["Period", "Geo Level", "State", "Domain", "Domain Category", "watershed_code"]

for col in EMPTY_COLS:
    assert raw[col].isna().all(), f"Expected {col!r} empty but it has values"
for col in CONST_COLS:
    assert raw[col].nunique(dropna=True) <= 1, f"Expected {col!r} constant: {raw[col].unique()}"

df = raw.drop(columns=EMPTY_COLS + CONST_COLS)
print(f"Dropped {len(EMPTY_COLS)} empty + {len(CONST_COLS)} constant columns; "
      f"{df.shape[1]} columns remain")

## Step 3 - Parse keys & values

- `year` -> integer.
- `county_fips` = 2-digit state ANSI + 3-digit county ANSI (zero-padded). The
  `OTHER COUNTIES` / `OTHER (COMBINED) COUNTIES` rollup buckets have no county
  ANSI, so their `county_fips` is left `NaN` (they are ag-district residuals,
  not real counties).
- `value` and `cv_pct` parsed from NASS text via the shared helper, with a
  `value_suppressed` flag preserving where a `(D)` was censored.

In [ ]:
df["year"] = df["Year"].astype(int)

state = df["State ANSI"].str.zfill(2)
county = df["County ANSI"].str.zfill(3)
df["county_fips"] = (state + county).where(df["County ANSI"].notna())

df["value"], df["value_suppressed"] = parse_nass_numeric(df["Value"])
df["cv_pct"], _ = parse_nass_numeric(df["CV (%)"])

print(f"county_fips built for {df['county_fips'].notna().sum():,} rows; "
      f"{df['county_fips'].isna().sum():,} rollup rows left NaN")
print(f"value: {df['value'].notna().sum():,} numeric, "
      f"{df['value_suppressed'].sum():,} suppressed (D)")

## Step 4 - Unpack `Data Item` into commodity / statistic / unit

`Data Item` packs three facts into one string. We split it, and keep the raw
`Commodity` column (renamed `commodity`) as the coarse grouping
(`CORN` covers both `CORN, GRAIN` and `CORN, SILAGE`).

In [ ]:
parsed = df["Data Item"].map(parse_data_item)
df["commodity_detail"] = parsed.map(lambda t: t[0])
df["statistic"] = parsed.map(lambda t: t[1])
df["unit"] = parsed.map(lambda t: t[2])

df = df.rename(columns={
    "Program": "program",
    "Commodity": "commodity",
    "County": "county",
    "Ag District": "ag_district",
    "Ag District Code": "ag_district_code",
})
print(df[["commodity", "commodity_detail", "statistic", "unit"]].drop_duplicates()
        .sort_values(["commodity", "statistic"]).to_string(index=False))

## Step 5 - Select, order, and enforce a unique key

The natural key is
`(program, year, ag_district_code, county, commodity_detail, statistic, unit)`.
`ag_district_code` is essential: the `OTHER (COMBINED) COUNTIES` bucket appears
once *per ag district*, so without it those rows collide.

In [ ]:
OUTPUT_COLS = [
    "program", "year",
    "ag_district_code", "ag_district", "county", "county_fips",
    "commodity", "commodity_detail", "statistic", "unit",
    "value", "value_suppressed", "cv_pct",
]
clean = df[OUTPUT_COLS].sort_values(
    ["program", "year", "ag_district_code", "county", "commodity_detail", "statistic"]
).reset_index(drop=True)

KEY = ["program", "year", "ag_district_code", "county", "commodity_detail", "statistic", "unit"]
dupes = clean.duplicated(KEY).sum()
assert dupes == 0, f"{dupes} duplicate key rows!"
print(f"Key is unique across {len(clean):,} rows.")
clean.head()

## Step 6 - Sanity check

In [ ]:
print(f"Rows: {len(clean):,}  |  years: {clean['year'].min()}-{clean['year'].max()}  |  "
      f"counties (with FIPS): {clean['county_fips'].nunique():,}")
print(f"programs: {clean['program'].value_counts().to_dict()}")
print(f"suppressed values: {clean['value_suppressed'].sum():,} "
      f"({clean['value_suppressed'].mean():.1%})\n")
print("Yields look physical (BU/ACRE etc.):")
display = clean[(clean.statistic == "YIELD") & clean.value.notna()]
print(display.groupby(["commodity_detail", "unit"])["value"].agg(["count", "min", "median", "max"]).to_string())

## Step 7 - Save

In [ ]:
out_file = CLEAN_DIR / "crop-yields-clean.csv"
clean.to_csv(out_file, index=False)
print(f"Saved {len(clean):,} rows -> {out_file}")